In [1]:
import torch
import torch.nn.functional as F

X = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y = torch.tensor([[0.0], [1.0]])

# Layer 1
W1 = torch.tensor([[-1.0, 2.0], [3.0, 0.5], [-0.1, -4.0]], requires_grad=True)
b1 = torch.tensor([1.0, -2.0, 0.3], requires_grad=True)

# Layer 2
W2 = torch.tensor([[0.5, 1.0, -2.0], [0.7, 0.1, 0.3]], requires_grad=True)
b2 = torch.tensor([-4.0, 5.0], requires_grad=True)

# Layer 3 (Output layer)
W3 = torch.tensor([[0.5, -0.3]], requires_grad=True)
b3 = torch.tensor([0.1], requires_grad=True)

In [2]:
Z1 = X @ W1.T + b1
A1 = torch.relu(Z1)
Z1.retain_grad()
A1.retain_grad()

Z2 = A1 @ W2.T + b2
A2 = torch.relu(Z2)
Z2.retain_grad()
A2.retain_grad()


Z3 = A2 @ W3.T + b3
A3 = F.sigmoid(Z3)
Z3.retain_grad()
A3.retain_grad()

cost = F.binary_cross_entropy(A3, y)
cost.retain_grad()

In [3]:
print(f"Z1:\n{Z1}")
print(f"A1:\n{A1}")
print(f"Z2:\n{Z2}")
print(f"A2:\n{A2}")
print(f"Z3:\n{Z3}")
print(f"A3:\n{A3}")
print(f"Cost:\n{cost}")

Z1:
tensor([[  4.0000,   2.0000,  -7.8000],
        [  6.0000,   9.0000, -16.0000]], grad_fn=<AddBackward0>)
A1:
tensor([[4., 2., 0.],
        [6., 9., 0.]], grad_fn=<ReluBackward0>)
Z2:
tensor([[ 0.0000,  8.0000],
        [ 8.0000, 10.1000]], grad_fn=<AddBackward0>)
A2:
tensor([[ 0.0000,  8.0000],
        [ 8.0000, 10.1000]], grad_fn=<ReluBackward0>)
Z3:
tensor([[-2.3000],
        [ 1.0700]], grad_fn=<AddBackward0>)
A3:
tensor([[0.0911],
        [0.7446]], grad_fn=<SigmoidBackward0>)
Cost:
0.19522885978221893


In [ ]:
import numpy as np

_A3 = A3.detach().numpy()
_A3[0, 0] = 1 - _A3[0, 0]
_L = -1 * np.log(_A3)
print(f"Log loss:\n{_L}")

Log loss:
[[0.09554543]
 [0.29491228]]


In [4]:
import torch.optim as optim

# Create a list of all parameters that need optimization
parameters = [W1, b1, W2, b2, W3, b3]

# Pass the parameter list to optimizer
optimizer = optim.SGD(parameters, lr=0.01)

# Then use the optimizer
optimizer.zero_grad()
cost.backward(retain_graph=True)

In [5]:
print(f"dJ/dL:\n{cost.grad}")
print(f"dJ/dA3:\n{A3.grad}")
print(f"dJ/dZ3:\n{Z3.grad}")
print(f"dJ/db3:\n{b3.grad}")
print(f"dJ/dW3:\n{W3.grad}")
print("-" * 50)
print(f"dJ/dA2:\n{A2.grad}")
print(f"dJ/dZ2:\n{Z2.grad}")
print(f"dJ/db2:\n{b2.grad}")
print(f"dJ/dW2:\n{W2.grad}")
print("-" * 50)
print(f"dJ/dA1:\n{A1.grad}")
print(f"dJ/dZ1:\n{Z1.grad}")
print(f"dJ/db1:\n{b1.grad}")
print(f"dJ/dW1:\n{W1.grad}")

dJ/dL:
1.0
dJ/dA3:
tensor([[ 0.5501],
        [-0.6715]])
dJ/dZ3:
tensor([[ 0.0456],
        [-0.1277]])
dJ/db3:
tensor([-0.0821])
dJ/dW3:
tensor([[-1.0216, -0.9253]])
--------------------------------------------------
dJ/dA2:
tensor([[ 0.0228, -0.0137],
        [-0.0639,  0.0383]])
dJ/dZ2:
tensor([[ 0.0000, -0.0137],
        [-0.0639,  0.0383]])
dJ/db2:
tensor([-0.0639,  0.0246])
dJ/dW2:
tensor([[-0.3831, -0.5747,  0.0000],
        [ 0.1752,  0.3175,  0.0000]])
--------------------------------------------------
dJ/dA1:
tensor([[-0.0096, -0.0014, -0.0041],
        [-0.0051, -0.0600,  0.1392]])
dJ/dZ1:
tensor([[-0.0096, -0.0014,  0.0000],
        [-0.0051, -0.0600,  0.0000]])
dJ/db1:
tensor([-0.0147, -0.0614,  0.0000])
dJ/dW1:
tensor([[-0.0249, -0.0396],
        [-0.1814, -0.2428],
        [ 0.0000,  0.0000]])


In [6]:
import numpy as np

# _dJ_Z2 = np.array([[0.0000, -0.0137, -0.0639, 0.0383]])
_dJ_Z1 = np.array([[-0.0096, -0.0014, 0.0000, -0.0051, -0.0600, 0.0000]])
_jacobian = np.array(
    [
        [1, 2, 0, 0, 0, 0],
        [0, 0, 1, 2, 0, 0],
        [1, 0, 0, 0, 1, 2],
        [3, 4, 0, 0, 0, 0],
        [0, 0, 3, 4, 0, 0],
        [0, 0, 0, 0, 3, 4],
    ]
)
print(_dJ_Z1 @ _jacobian)
print(f"Z1\n{Z1}")
print(f"X\n{X}")

[[-0.0249 -0.0396 -0.1814 -0.2428  0.      0.    ]]
Z1
tensor([[  4.0000,   2.0000,  -7.8000],
        [  6.0000,   9.0000, -16.0000]], grad_fn=<AddBackward0>)
X
tensor([[1., 2.],
        [3., 4.]])


In [7]:
print(f"W1\n{W1}")
print(f"W1.grad\n{W1.grad}")
print(f"b1\n{b1}")
print(f"b1.grad\n{b1.grad}")

optimizer.step()

print(f"Updated W1:\n{W1}")
print(f"Updated b1:\n{b1}")

W1
tensor([[-1.0000,  2.0000],
        [ 3.0000,  0.5000],
        [-0.1000, -4.0000]], requires_grad=True)
W1.grad
tensor([[-0.0249, -0.0396],
        [-0.1814, -0.2428],
        [ 0.0000,  0.0000]])
b1
tensor([ 1.0000, -2.0000,  0.3000], requires_grad=True)
b1.grad
tensor([-0.0147, -0.0614,  0.0000])
Updated W1:
tensor([[-0.9998,  2.0004],
        [ 3.0018,  0.5024],
        [-0.1000, -4.0000]], requires_grad=True)
Updated b1:
tensor([ 1.0001, -1.9994,  0.3000], requires_grad=True)


In [ ]:
_b1 = np.array([1.0000, -2.0000, 0.3000])
_b1_grad = np.array([-0.0147, -0.0614, 0.0000])
learning_rate = 0.01

_updated_b1 = _b1 - learning_rate * _b1_grad
print(f"Manually calculated Updated b1:\n{_updated_b1}")
print(f"Updated b1:\n{b1}")

Manually calculated Updated b1:
[ 1.000147 -1.999386  0.3     ]
Updated b1:
tensor([ 1.0001, -1.9994,  0.3000], requires_grad=True)
